# RooFit Tutorial: Introduction to Unbinned and Binned Likelihood Models

## Setup

Import ROOT and NumPy:

In [ ]:
import ROOT
import numpy as np

Silence the RooFit logging:

In [ ]:
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.FATAL)

## The basics

Mathematical concepts are represented by C++ objects:

In [ ]:
from IPython.display import Image, display
display(Image(filename="../images/roofit_classes.png"))

### Creating your first RooFit model

In [ ]:
import ROOT
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.FATAL)

Observable:

In [ ]:
x = ROOT.RooRealVar("x", "x", 0, 0, 10)

Parameters:

In [ ]:
mean = ROOT.RooRealVar("mean", "mean of gaussian", 5, 0, 10)
sigma = ROOT.RooRealVar("sigma", "width of gaussian", 1, 0.1, 10)

Gaussian PDF (**left as exercise**, *see the collapsed cell for solution*):

In [ ]:
# Define the a Gaussian pdf as a function of x, mean, and sigma here
# gauss = ??

In [ ]:
# Here is the solution!
gauss = ROOT.RooGaussian("gauss", "gaussian PDF", x, mean, sigma)

PDF inspection:

In [ ]:
gauss.Print("t")

### Toy dataset generation and fitting

Generate a toy dataset with 9000 entries sampled from the Gaussian PDF:

In [ ]:
data = gauss.generate({x}, 9000)

In [ ]:
data.Print()

Fit the PDF to the toy data, saving the fit result:

In [ ]:
fit_result = gauss.fitTo(data, PrintLevel=-1, Save=True)

In [ ]:
fit_result.Print()

Inspect the correlation of your model parameters:

In [ ]:
fit_result.correlationMatrix().Print()

## Plotting the data and the model

Create a `RooPlot` object on which the data and PDF is plotted:

In [ ]:
x_frame = x.frame(Title="Gaussian PDF with data")

In [ ]:
data.plotOn(x_frame)
gauss.plotOn(x_frame);

Draw the RooPlot on a `TCanvas`:

In [ ]:
c1 = ROOT.TCanvas("c1", "c1", 500, 300)
x_frame.Draw()
c1.Draw()

### Importing RooFit datasets from a ROOT file

Export the dataset to a ROOT file so we can show how to import it:

In [ ]:
# The content of this cell doesn't matter, it's just producing the input file
data.convertToTreeStore()

output_file = ROOT.TFile("dataset.root", "RECREATE")
data.store().tree().SetName("mytree")
data.store().tree().SetTitle("My measured data")
data.store().tree().Write()
output_file.Close()

data.convertToVectorStore()

ROOT file with a TTree that stores the data you want to fit:

In [ ]:
input_file = ROOT.TFile("dataset.root", "READ") # file contains a TTree called "mytree"

Import to `RooDataSet` using the constructor that takes a TTree:

In [ ]:
dataset = ROOT.RooDataSet("dataset", "dataset", x, Import=input_file["mytree"])

Don't forget to close the file:

In [ ]:
input_file.Close()

You will see again the dataset with the observable `x`:

In [ ]:
dataset.Print()

### Exporting your RooFit datasets

You can export a RooDataSet to NumPy or Pandas:

In [ ]:
df = data.to_pandas()

In [ ]:
df

## Composite PDFs

Composite PDF: model with mutiple components, like signal and background.

This time, we create a toy data set with RDataFrame, like you would in a real analysis:

In [ ]:
df = ROOT.RDataFrame(45000).Define(
    "x",
    "rdfentry_ < 9000 ? gRandom->Gaus(5.0, 1.0) : gRandom->Exp(1.0 / 0.18)"
)
print("Total entries:", df.Count().GetValue())

Create the RooDataSet via a RDataFrame helper action (as explained in [this tutorial](https://root.cern/doc/master/rf408__RDataFrameToRooFit_8py.html)):

In [ ]:
data_x = df.Book(
    ROOT.std.move(ROOT.RooDataSetHelper("my_data", "", ROOT.RooArgSet(x))), ("x",)
).GetValue()
data_x.Print() # events outside RooRealVar range will be dropped

Visualize the dataset:

In [ ]:
x_frame = x.frame(Title="Plotting Gaussian plus exp. background")
data_x.plotOn(x_frame)

c2 = ROOT.TCanvas()
x_frame.Draw()
c2.Draw()

### Creating the composite fit model

Create exponential PDF with parameter "tau":

In [ ]:
tau = ROOT.RooRealVar("tau", "tau", -0.2, -10.0, -0.01)
expo = ROOT.RooExponential("expo", "expo", x, tau)

Define parameters for the number of signal and background events:

In [ ]:
n_sig = ROOT.RooRealVar("n_sig", "n_sig", 10000, 1000, 100000)
n_bkg = ROOT.RooRealVar("n_bkg", "n_bkg", 50000, 5000, 500000)

Composite model that automatically includes a Poisson term for the total number of events:

**Exercise:** try to write the [RooAddPdf](https://root.cern.ch/doc/master/classRooAddPdf.html) that defines the model yourself, and assign it to a variable called `model`. Look at the linked documentation to find the appropriate constructor.

In [ ]:
# write code in this cell..
# model = ??

In [ ]:
# Solution here:
model = ROOT.RooAddPdf("model", "model", [gauss, expo], [n_sig, n_bkg])

Do the fit:

In [ ]:
fit_result = model.fitTo(data_x, PrintLevel=-1, Save=True)
fit_result.Print()

## Creating a nice plot

Create RooPlot and draw data, PDF, and components:

In [ ]:
x_frame = x.frame(Title="Gaussian plus exp. background")

data_x.plotOn(x_frame, Name="data")

model.plotOn(x_frame, Components=gauss, LineColor="r", LineStyle="--", Name="gauss")
model.plotOn(x_frame, Components=expo, LineColor="k", LineStyle="--", Name="expo")
model.plotOn(x_frame, Name="model");

Add a legend:

In [ ]:
# do you really want to see this boilerplate code... ?
legend = ROOT.TLegend(0.7, 0.55, 0.92, 0.87)
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.AddEntry(x_frame.findObject("data"), "data", "P")

for name in ["model", "gauss", "expo"]:
    legend.AddEntry(x_frame.findObject(name), name, "L")

Create a second frame with the residuals:

In [ ]:
resid_hist = x_frame.residHist()

resid_frame = x.frame(Title=";x;residuals")
resid_frame.addPlotable(resid_hist, "P")

Create a canvas that is divided into two drawing pads:

In [ ]:
c3 = ROOT.TCanvas("c3", "c3", 600, 600)
c3.Divide(1, 2)

First pad is for the main plot and the legend:

In [ ]:
pad_1 = c3.cd(1)
x_frame.Draw()
legend.Draw()
pad_1.SetPad(0.0, 0.2, 1, 1)

Second pad is for the residuals:

In [ ]:
pad_2 = c3.cd(2)
pad_2.SetPad(0., 0.0, 1, 0.25)
resid_frame.Draw()
resid_frame.GetXaxis().SetLabelSize(0.12)
resid_frame.GetYaxis().SetLabelSize(0.12)
resid_frame.GetYaxis().SetTitleSize(0.12)
resid_frame.GetYaxis().SetTitleOffset(0.25)

Draw the canvas:

In [ ]:
c3.Draw()

**Exerciese task:** Further improve the plot with the pull distribution by visualizing also the post-fit uncertainty of the model. Figure out how to do this by reading the documentation of [RooAbsPdf::plotOn()](https://root.cern.ch/doc/master/classRooAbsPdf.html#aa0f2f98d89525302a06a1b7f1b0c2aa6).

## Template fits with convolutions

A **template** PDF is based on *histogram shape*, and not expressed by an analytical function.

Imagine you have a histogram giving the expected signal shape:

In [ ]:
template_hist = ROOT.TH1D("h1", "h1", 100, 0, 10)
f1 = ROOT.TF1("f1", "std::exp(-std::abs((x-5)))", 0, 10)
template_hist.FillRandom("f1", 100000)

In [ ]:
c4 = ROOT.TCanvas("c4", "c4", 600, 400)
template_hist.Draw()
c4.Draw()

Getting the template PDF into RooFit:

1. Create a corresponding observable:

In [ ]:
y = ROOT.RooRealVar("y", "y", 0, 0, 10)

2. Convert the `TH1` into a `RooDataHist`:

In [ ]:
roo_template_hist = ROOT.RooDataHist("roo_template_hist", "roo_template_hist", y, template_hist)

3. Create a `RooHistPdf` based of the RooFit histogram:

In [ ]:
sig_raw_y = ROOT.RooHistPdf("sig__raw_y", "sig_raw_y", y, roo_template_hist)

### Creating a full composite model

Construct a `RooGaussian` to model detector resolution effects:

In [ ]:
resolution = ROOT.RooRealVar("resolution", "resolution", 0.2, 0.1, 1.0)
sig_smearing_y = ROOT.RooGaussian("sig_smearing_y", "sig_smearing_y", y, ROOT.RooFit.RooConst(0.0), resolution)

The signal PDF is a convolution of the template and the resolution function:

**Exercise:** define a [RooFFTConvPdf](https://root.cern.ch/doc/master/classRooFFTConvPdf.html) and assign to the `sig_y` variable. It should be the convolution of the `sig_raw_y` template and the `sig_smearing_y` resolution function:

In [ ]:
# define the RooFFTConvPdf here:
# sig_y = ???

In [ ]:
# Solution code here:
sig_y = ROOT.RooFFTConvPdf("sig_y", "sig_y", y, sig_raw_y, sig_smearing_y)

For the background, we use a Chebychev polynomial:

In [ ]:
bkg_y = ROOT.RooChebychev("bkg_y", "bkg_y", y, [-0.5, 0.1])

Finally, create **RooAddPdf** for the composite model:

In [ ]:
model_y = ROOT.RooAddPdf("model_y", "model_x", [sig_y, bkg_y], [n_sig, n_bkg])

Creating toy dataset and fitting

In [ ]:
data_y = model_y.generate(y)

In [ ]:
fit_result = model_y.fitTo(data_y, PrintLevel=-1, Save=True)
fit_result.Print()

Plotting the model and the toy dataset

In [ ]:
y_frame = y.frame(Title="Model for y")

data_y.plotOn(y_frame)
model_y.plotOn(y_frame)

c5 = ROOT.TCanvas()
y_frame.Draw()
c5.Draw()

**Exercise task:** Look at the [rf203_ranges.py RooFit tutorial](https://root.cern/doc/master/rf203__ranges_8py.html) to learn how to restrict the fit to a subrange. Redo the convoluted template fit to the $y$ variable, but restricted to the range from 3 to 7.

   Why does the uncertainty of the `resolution` parameter increase, even though we are not excluding that much signal and `resolution` doesn't affect the background?

### Overview of other PDF types

RooFit provides a collection of standard PDF classes, e.g.:

In [ ]:
from IPython.display import Image, display
display(Image(filename="../images/roofit_pdfs.png"))

Easy to **extend the library**: each pdf is a separate C++ class

## Multivariate fit

We have now modeled two observables:
* `x` with Gaussian signal and exponential background
* `y` with smeared template signal and Chebychev background

Create 2D model $p(x,y) = p(x) p(y)$ for signal and background

In [ ]:
model_sig_xy = ROOT.RooProdPdf("model_sig_xy", "model_sig_xy", [gauss, sig_y])
model_bkg_xy = ROOT.RooProdPdf("model_bkg_xy", "model_bkg_xy", [expo, bkg_y])

Yet again, a RooAddPdf for the final model

In [ ]:
model_xy = ROOT.RooAddPdf("model_xy", "model_xy", [model_sig_xy, model_bkg_xy], [n_sig, n_bkg])

Generating a 2D toy dataset

In [ ]:
data_xy = model_xy.generate({x, y}, 10000)

Visualize the 2D data in a LEGO plot

In [ ]:
histo_xy = data_xy.createHistogram("histo_xy", x, Binning=25, YVar=dict(var=data_xy.get()["y"], Binning=15))
histo_xy.SetTitle("")

c6 = ROOT.TCanvas()
histo_xy.Draw("LEGO2")
c6.Draw()

Fitting the 2D model

By now you know how it works:

In [ ]:
fit_result_xy = model_xy.fitTo(data_xy, PrintLevel=-1, Save=True)

Fit result has all parameters we saw before:

In [ ]:
fit_result_xy.Print()

**Exercise question:** Which parameters are strongly (anti)correlated in the final 2D fit? Can you explain why?

### 1D-projected visualizations of 2D model and data

In [ ]:
x_frame = x.frame(Title="Model for x")
y_frame = y.frame(Title="Model for y")

data_xy.plotOn(x_frame)
model_xy.plotOn(x_frame)

data_xy.plotOn(y_frame)
model_xy.plotOn(y_frame)

c7 = ROOT.TCanvas("c7", "c7", 800, 400)
c7.Divide(2)
c7.cd(1)
x_frame.Draw()
c7.cd(2)
y_frame.Draw()
c7.Draw()

**Exercise question:** 6. For the multidimensional model, why did we not just create a single `RooProdPdf` that multiples the model for $x$ and the model for $y$?

### Likelihood scans

It is very useful to plot and inspect the NLL and also the profiled NLL. For this, you can use `createNLL` to get a RooFit object that represents a likelihood directly. More examples can be found in the [rf605_profilell tutorial](https://root.cern/doc/master/rf605__profilell_8py.html).

In [ ]:
# Create likelihood function
nll = model_xy.createNLL(data_xy, EvalBackend="cpu")
# The new "cpu" evaluation backend is a performance optimization, it can also be used in `fitTo`

# Minimize likelihood such that all other parameters (nuisance parameters) are at the best fit value.
minimizer = ROOT.RooMinimizer(nll)
minimizer.setPrintLevel(-1)
minimizer.minimize("Minuit", "")

# Make RooPlot for our parameter of interest, let's say n_sig
window = 5 * n_sig.getError()
n_sig_frame = n_sig.frame(Bins=10, Range=(n_sig.getVal() - window, n_sig.getVal() + window))

# Plot likelihood scan in parameter n_sig
nll.plotOn(n_sig_frame, ShiftToZero=True)

# Plot the profile likelihood in n_sig.
# Now, the nuisance parameter are optimized for each scanned value of n_sig.
pll_n_sig = nll.createProfile([n_sig])
pll_n_sig.plotOn(n_sig_frame, LineColor="kRed")

# Set y axis limits
n_sig_frame.SetMinimum(0)
n_sig_frame.SetMaximum(5)

c8 = ROOT.TCanvas()
n_sig_frame.Draw()
c8.Draw()

**Exercise question 1:** `n_sig` is the parameter of interest for this likelihood scan. What are the nuisance parameters?

**Exercise question 2:** Why is the profile NLL always below the other plotted NLL?

## Model inspection

You already know the `Print("t")` function:

In [ ]:
model.Print("t")

## The RooWorkspace

The RooFit objects can be managed by a `RooWorkspace`:

In [ ]:
ws = ROOT.RooWorkspace("myworkspace")

You can for example import an existing model:

In [ ]:
ws.Import(model_xy);

You can `Print` the workspace for inspecting its content:

In [ ]:
ws.Print()

Access any object in the RooWorkspace:

In [ ]:
ws["model_xy"].Print()

Save the workspace to a ROOT file to reuse the model later

In [ ]:
ws.writeToFile("myworkspace.root");

### Serialization to HS3-compliant JSON

We can also serialize the workspace to a JSON file that follows the [**HEP Statistics Serialization Standard (HS3)**](https://hep-statistics-serialization-standard.github.io/), using the [RooJSONFactoryWSTool](https://root.cern/doc/v636/classRooJSONFactoryWSTool.html) *(yes the name makes no sense and might change)*:

In [ ]:
tool = ROOT.RooJSONFactoryWSTool(ws)
tool.exportJSON("workspace.json");

Let's look a the exported JSON:

In [ ]:
!cat workspace.json | python3 -m json.tool

**Exersice task:** import the JSON to a fresh RooWorkspace and try to make some closure check to validate that the round-tripped workspace is the same.

## Binned models and HistFactory

So far, we did **unbinned** fits with models that have an analytical shape (possibly convoluted with a template).

In LHC analyses, one often performs **binned** likelihood fits instead: the model predicts the expected number of events in each bin of a histogram, and the prediction for each sample is usually taken from a **Monte Carlo template histogram**.

Systematic uncertainties are implemented with **nuisance parameters** that continuously *interpolate* between the nominal template and systematically varied templates, changing the normalization or even the shape of the prediction. Each nuisance parameter is accompanied by a constraint term in the likelihood.

Even for a single channel with a few samples, spelling out such a likelihood with individual RooFit objects is tedious and error-prone. Therefore, binned models are usually created with **higher-level frameworks on top of RooFit**. The framework that ships with ROOT is [HistFactory](https://root.cern/doc/master/group__HistFactory.html).

We will now build the model from the [hf001_example.py tutorial](https://root.cern/doc/master/hf001__example_8py.html): one channel with a signal sample and two background samples.

First, we create the template and data histograms and save them to a ROOT file, just like the histogram-making step of a real analysis would:

In [ ]:
# Just in case you want to re-start the tutorial from here:
import ROOT
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.FATAL)

In [ ]:
input_file_name = "hf_input.root"

h_sig = ROOT.TH1D("signal", "signal template", 2, 1, 2)
h_sig.SetBinContent(1, 20)
h_sig.SetBinContent(2, 10)

h_bkg1 = ROOT.TH1D("background1", "background 1 template", 2, 1, 2)
h_bkg1.SetBinContent(1, 100)

h_bkg2 = ROOT.TH1D("background2", "background 2 template", 2, 1, 2)
h_bkg2.SetBinContent(2, 100)

# Relative statistical uncertainties of the background 1 template:
h_bkg1_statuncert = ROOT.TH1D("background1_statUncert", "background 1 rel. uncert.", 2, 1, 2)
h_bkg1_statuncert.SetBinContent(1, 0.05)
h_bkg1_statuncert.SetBinContent(2, 0.05)

# The observed data:
h_data = ROOT.TH1D("data", "data", 2, 1, 2)
h_data.SetBinContent(1, 122)
h_data.SetBinContent(2, 112)

with ROOT.TFile.Open(input_file_name, "RECREATE") as hf_input_file:
    for hist in [h_sig, h_bkg1, h_bkg2, h_bkg1_statuncert, h_data]:
        hf_input_file.WriteObject(hist, hist.GetName())

**Exercise question:** Can you explain the meaning of all of these histograms? In particular, what could be the physics analysis motivation for `background1_statUncert`? Isn't the statistical uncertainty implicitly the square root of the bin content, which would be $\sqrt(100) = 10$, so 10 % of the bin content?

A HistFactory model is declared with a `Measurement` object. We define the parameter of interest (the signal strength `SigXsecOverSM`) and the integrated luminosity with its uncertainty. Like in the original tutorial, the luminosity and the nuisance parameter for the signal systematic are kept constant in the fit:

In [ ]:
meas = ROOT.RooStats.HistFactory.Measurement("meas", "meas")

meas.SetPOI("SigXsecOverSM")
meas.SetLumi(1.0)
meas.SetLumiRelErr(0.10)

meas.AddConstantParam("Lumi")
meas.AddConstantParam("alpha_syst1")

A measurement contains one or more **channels**, i.e. disjoint regions of the data like signal or control regions. Each channel gets its observed data and a configuration for the statistical uncertainties of the templates (here: ignore them below 2 % relative uncertainty, and use Poisson constraint terms):

In [ ]:
chan = ROOT.RooStats.HistFactory.Channel("channel1")
chan.SetData("data", input_file_name)
chan.SetStatErrorConfig(0.02, "Poisson")

Each channel contains **samples**, whose expected distributions are given by the template histograms. The signal sample gets a free normalization factor (our parameter of interest) and a ±5 % normalization uncertainty called `syst1`:

In [ ]:
signal = ROOT.RooStats.HistFactory.Sample("signal", "signal", input_file_name)
signal.AddOverallSys("syst1", 0.95, 1.05)
signal.AddNormFactor("SigXsecOverSM", 1, 0, 3)
chan.AddSample(signal)

The background samples get normalization uncertainties too. In addition, we activate the **statistical uncertainty of the templates** themselves: for `background1` it is read from the dedicated histogram of relative uncertainties we created above, and for `background2` it is taken from the bin errors of the template. HistFactory turns this into one nuisance parameter per bin, shared by all samples in the channel and constrained by the configured Poisson terms (also known as the *Barlow–Beeston method*).

If you also want a systematic variation to change the *shape* of a template, you would use `AddHistoSys()`, which takes a down- and up-varied histogram to interpolate between — see the [HistFactory documentation](https://root.cern/doc/master/group__HistFactory.html).

In [ ]:
background1 = ROOT.RooStats.HistFactory.Sample("background1", "background1", input_file_name)
background1.ActivateStatError("background1_statUncert", input_file_name)
background1.AddOverallSys("syst2", 0.95, 1.05)
chan.AddSample(background1)

background2 = ROOT.RooStats.HistFactory.Sample("background2", "background2", input_file_name)
background2.ActivateStatError()
background2.AddOverallSys("syst3", 0.95, 1.05)
chan.AddSample(background2)

Add the channel to the measurement and collect the histograms from the input file:

In [ ]:
meas.AddChannel(chan)
meas.CollectHistograms();

So far, everything was purely *declarative* and no RooFit objects were involved yet. Now we let HistFactory build the actual model from the measurement specification:

In [ ]:
ws_hf = ROOT.RooStats.HistFactory.HistoToWorkspaceFactoryFast.MakeCombinedModel(meas)

The result is a `RooWorkspace`, which you already know. Printing it shows how much work HistFactory did for us: the templates became `RooHistFunc` objects, the systematics became `FlexibleInterpVar` interpolation objects with `RooGaussian` constraint terms, the template statistics became one `gamma_stat_*` parameter per bin with Poisson constraints, and everything is combined into a `RooSimultaneous` PDF:

In [ ]:
ws_hf.Print()

The workspace also contains a `ModelConfig` object that documents the model for the statistics tools in RooStats: which PDF to use, which parameters are the parameters of interest, which are the observables, and which are the *global observables* of the constraint terms:

In [ ]:
model_config = ws_hf["ModelConfig"]
pdf_hf = model_config.GetPdf()

Since this is an ordinary RooFit PDF, we can fit it to the observed data like any other model. For models with constraint terms, it is recommended to explicitly pass the global observables:

In [ ]:
# Remember the pre-fit values and constraint widths of all floating
# nuisance parameters: fitTo() moves the workspace parameters to the
# post-fit point (and their errors to the post-fit uncertainties), and
# we need the pre-fit state for the fit-quality and pulls plots below.
prefit_pars = {
    par.GetName(): (par.getVal(), par.getError())
    for par in model_config.GetNuisanceParameters()
    if not par.isConstant()
}

fit_result_hf = pdf_hf.fitTo(
    ws_hf["obsData"],
    GlobalObservables=model_config.GetGlobalObservables(),
    PrintLevel=-1,
    Save=True,
)
fit_result_hf.Print()

**Exercise question:** Let's take a step back and try to understand the result. What is the meaning of all these parameters? What are the parameters of interest, and what are the nuisance parameters?

All the tools from before work here as well, for example the profile likelihood scan of the signal strength, where all the nuisance parameters that HistFactory created are profiled:

In [ ]:
poi = model_config.GetParametersOfInterest().first()

nll_hf = pdf_hf.createNLL(ws_hf["obsData"], EvalBackend="cpu")
pll_hf = nll_hf.createProfile([poi])

poi_frame = poi.frame(Title="Profile likelihood scan of the signal strength")
nll_hf.plotOn(poi_frame, ShiftToZero=True, LineColor="kRed", LineStyle="--")
pll_hf.plotOn(poi_frame)
poi_frame.GetYaxis().SetTitle("-log likelihood")
poi_frame.SetMinimum(0)
poi_frame.SetMaximum(3)

c9 = ROOT.TCanvas("c9", "c9", 500, 300)
poi_frame.Draw()
c9.Draw()

**Exercise task:** In the HistFactory example, the nuisance parameter `alpha_syst1` of the signal normalization uncertainty was set constant. Remove the corresponding `AddConstantParam()` call and rebuild the model. How do the fitted value and the uncertainty of `SigXsecOverSM` change? What happens if you increase `syst1` to a ±20 % uncertainty?

### Visualize fit results and pulls

Pre- and post-fit histograms:

In [ ]:
# Nothing to see here except for extremely verbose plotting code...

# -------------------------------------------------
# Pre- and post-fit fit-quality plots of the HistFactory model
#
# Both panels show the stacked expectation of the template samples
# with the data (Poisson error bars), and a hatched grey 1-sigma
# band for the model uncertainty, propagated bin-by-bin:
#
#   pre-fit  (left) : diagonal propagation of the independent pre-fit
#                     constraint widths of the nuisance parameters
#                     (unit Gaussians for the alpha_* OverallSys
#                     parameters, relative MC-stat. uncertainties for
#                     the gamma_stat_ parameters). The POI is
#                     unconstrained pre-fit and contributes nothing.
#
#   post-fit (right): linear propagation of the FULL post-fit
#                     covariance matrix of fit_result_hf, including
#                     the correlations that the fit established.
#
# The bottom sub-panels show the residuals in units of the Poisson
# data uncertainty.
# -------------------------------------------------
import math

channel_model = ws_hf["channel1_model"]  # RooRealSumPdf of the channel
obs_hf = ws_hf["obs_x_channel1"]  # the binned observable
obs_binning = obs_hf.getBinning()
hf_nbins = obs_binning.numBins()
hf_data = ws_hf["obsData"]
sample_order = ["background1", "background2", "signal"]  # stack bottom to top
sample_colors = {
    "background1": ROOT.kAzure - 4,
    "background2": ROOT.kGreen + 2,
    "signal": ROOT.kOrange + 1,
}


def model_bin_contents():
    """Expected counts of the full channel model at the CURRENT parameter values."""
    out = []
    for i in range(hf_nbins):
        obs_hf.setVal(obs_binning.binCenter(i))
        # channel_model is a bin density: counts = density x bin width
        out.append(channel_model.getVal() * obs_binning.binWidth(i))
    return out


def sample_bin_contents(sample_name):
    """Expected counts of one sample: coefficient times shape of its
    RooRealSumPdf component."""
    out = [0.0] * hf_nbins
    for func, coef in zip(channel_model.funcList(), channel_model.coefList()):
        if not func.GetName().startswith(sample_name + "_"):
            continue
        for i in range(hf_nbins):
            obs_hf.setVal(obs_binning.binCenter(i))
            out[i] += func.getVal() * coef.getVal() * obs_binning.binWidth(i)
    return out


def data_bin_contents():
    """Observed counts per bin."""
    out = []
    for i in range(hf_data.numEntries()):
        hf_data.get(i)
        out.append(hf_data.weight())
    return out


def prefit_band():
    """1-sigma band from the independent ('diagonal') pre-fit constraint
    widths. The POI is unconstrained pre-fit and contributes nothing."""
    variance = [0.0] * hf_nbins
    for name, (val0, sigma0) in prefit_pars.items():
        if sigma0 <= 0.0:
            continue
        par = ws_hf[name]
        par.setVal(val0 + sigma0)
        up = model_bin_contents()
        par.setVal(val0 - sigma0)
        down = model_bin_contents()
        par.setVal(val0)
        for i in range(hf_nbins):
            variance[i] += (0.5 * (up[i] - down[i])) ** 2
    return [math.sqrt(v) for v in variance]


def postfit_band():
    """1-sigma band from linear propagation of the FULL post-fit covariance
    matrix (including correlations) of the fit result."""
    covariance = fit_result_hf.covarianceMatrix()
    gradients = []
    for fit_par in fit_result_hf.floatParsFinal():
        par = ws_hf[fit_par.GetName()]
        step = max(1e-4, 1e-2 * par.getError())
        val0 = par.getVal()
        par.setVal(val0 + step)
        up = model_bin_contents()
        par.setVal(val0 - step)
        down = model_bin_contents()
        par.setVal(val0)
        gradients.append([(u - d) / (2 * step) for u, d in zip(up, down)])

    band = []
    for i in range(hf_nbins):
        var = sum(gradients[p][i] * covariance[p][q] * gradients[q][i]
                  for p in range(len(gradients)) for q in range(len(gradients)))
        band.append(math.sqrt(max(var, 0.0)))
    return band


data_contents = data_bin_contents()
poi_var = ws_hf[model_config.GetParametersOfInterest().first().GetName()]

# --- move parameters to the pre-fit point and evaluate ---------------------
for name, (val0, _sigma0) in prefit_pars.items():
    ws_hf[name].setVal(val0)
poi_var.setVal(1.0)  # nominal value of the SigXsecOverSM NormFactor

prefit_total = model_bin_contents()
prefit_samples = {s: sample_bin_contents(s) for s in sample_order}
prefit_sigma = prefit_band()

# --- move to the post-fit point and evaluate -------------------------------
for fit_par in fit_result_hf.floatParsFinal():
    ws_hf[fit_par.GetName()].setVal(fit_par.getVal())
    ws_hf[fit_par.GetName()].setError(fit_par.getError())

postfit_total = model_bin_contents()
postfit_samples = {s: sample_bin_contents(s) for s in sample_order}
postfit_sigma = postfit_band()

print("prefit  total:", [round(v, 2) for v in prefit_total],
      "band:", [round(v, 2) for v in prefit_sigma])
print("postfit total:", [round(v, 2) for v in postfit_total],
      "band:", [round(v, 2) for v in postfit_sigma])
print("postfit samples:", {s: [round(v, 2) for v in c] for s, c in postfit_samples.items()})

# --- drawing helpers --------------------------------------------------------
persistent_objects = []  # keep PyROOT wrappers alive


def make_h1(name, title, contents):
    hist = ROOT.TH1D(name, title, hf_nbins, obs_binning.lowBound(), obs_binning.highBound())
    for i in range(hf_nbins):
        hist.SetBinContent(i + 1, contents[i])
    persistent_objects.append(hist)
    return hist


def make_band_graph(total, sigma):
    g = ROOT.TGraphAsymmErrors(hf_nbins)
    for i in range(hf_nbins):
        g.SetPoint(i, obs_binning.binCenter(i), total[i])
        g.SetPointError(i, 0.5 * obs_binning.binWidth(i), 0.5 * obs_binning.binWidth(i),
                        sigma[i], sigma[i])
    g.SetFillColor(ROOT.kGray + 1)
    g.SetFillStyle(3354)  # dense diagonal hatching
    persistent_objects.append(g)
    return g


def draw_fit_panel(pad_top, pad_bottom, title, total, samples, sigma, header,
                   residual_title="Residual"):
    """Stacked data-vs-model plot with hatched model-uncertainty band and a
    residuals sub-panel."""
    pad_top.cd()

    cumulative = [0.0] * hf_nbins
    cumulative_hists = []
    for sample in sample_order:
        cumulative = [c + s for c, s in zip(cumulative, samples[sample])]
        hist = make_h1(f"h_{sample}_{title}", "", cumulative)
        hist.SetFillColor(sample_colors[sample])
        hist.SetLineColor(sample_colors[sample])
        cumulative_hists.append((sample, hist))

    h_total_line = make_h1(f"h_tot_{title}", "", total)
    h_total_line.SetLineColor(ROOT.kBlack)
    h_total_line.SetLineWidth(2)

    h_data = make_h1(f"h_data_{title}", title, data_contents)
    h_data.SetBinErrorOption(ROOT.TH1.kPoisson)  # asymmetric Poisson error bars
    h_data.SetMarkerStyle(20)
    h_data.SetLineColor(ROOT.kBlack)
    h_data.SetLineWidth(2)

    band_graph = make_band_graph(total, sigma)

    ymax = 1.5 * max(max(d + math.sqrt(d) for d in data_contents),
                     max(t + s for t, s in zip(total, sigma)))

    # the frame is carried by the data histogram to get a meaningful minimum
    h_data.GetXaxis().SetLabelSize(0)  # x labels only on the bottom panel
    h_data.GetYaxis().SetTitle("Events")
    h_data.GetYaxis().SetTitleSize(0.062)
    h_data.GetYaxis().SetTitleOffset(0.85)
    h_data.GetYaxis().SetLabelSize(0.055)
    h_data.SetMaximum(ymax)
    h_data.SetMinimum(0.0)
    h_data.Draw("E0 X0")
    for _sample, hist in reversed(cumulative_hists):
        hist.Draw("hist same")
    band_graph.Draw("2 same")
    #h_total_line.Draw("hist same")
    h_data.Draw("E0 X0 same")

    legend = ROOT.TLegend(0.45, 0.62, 0.90, 0.92)
    persistent_objects.append(legend)
    legend.SetBorderSize(0)
    legend.SetFillStyle(0)
    legend.SetTextSize(0.055)
    legend.SetHeader(header, "C")
    legend.AddEntry(h_data, "Data", "PE")
    for sample in reversed(sample_order):
        pretty = {"background1": "Background 1", "background2": "Background 2",
                  "signal": "Signal (#mu #times s)"}[sample]
        legend.AddEntry(dict(cumulative_hists)[sample], pretty, "F")
    legend.AddEntry(band_graph, "Model unc. (1#sigma)", "F")
    legend.Draw()

    # bottom panel: residuals in units of the Poisson data uncertainty
    pad_bottom.cd()
    h_res = make_h1(f"h_res_{title}", "", [0.0] * hf_nbins)
    g_res_band = ROOT.TGraphAsymmErrors(hf_nbins)
    persistent_objects.append(g_res_band)
    max_res = 3.4
    for i in range(hf_nbins):
        if data_contents[i] <= 0.0:
            continue
        sigma_data = math.sqrt(data_contents[i])
        res = (data_contents[i] - total[i]) / sigma_data
        h_res.SetBinContent(i + 1, res)
        h_res.SetBinError(i + 1, 1.0)  # +-1 sigma of the data
        g_res_band.SetPoint(i, obs_binning.binCenter(i), 0.0)
        g_res_band.SetPointError(i, 0.5 * obs_binning.binWidth(i),
                                 0.5 * obs_binning.binWidth(i),
                                 sigma[i] / sigma_data, sigma[i] / sigma_data)
        max_res = max(max_res, 1.15 * max(abs(res), sigma[i] / sigma_data))

    g_res_band.SetFillColor(ROOT.kGray + 1)
    g_res_band.SetFillStyle(3354)
    h_res.SetMarkerStyle(20)
    h_res.SetMarkerSize(0.9)
    h_res.SetLineColor(ROOT.kBlack)
    h_res.SetLineWidth(2)

    h_res.GetYaxis().SetRangeUser(-max_res, max_res)
    h_res.GetYaxis().SetNdivisions(305)
    h_res.GetYaxis().SetTitle(residual_title)
    h_res.GetYaxis().CenterTitle()
    h_res.GetYaxis().SetTitleSize(0.14)
    h_res.GetYaxis().SetTitleOffset(0.30)
    h_res.GetYaxis().SetLabelSize(0.11)
    h_res.GetXaxis().SetTitle("channel1 bin")
    h_res.GetXaxis().SetTitleSize(0.14)
    h_res.GetXaxis().SetTitleOffset(0.95)
    h_res.GetXaxis().SetLabelSize(0.11)
    for i in range(hf_nbins):  # label the bins by their number
        h_res.GetXaxis().SetBinLabel(i + 1, f"bin {i + 1}")
    h_res.GetXaxis().CenterLabels()
    h_res.Draw("PE")
    g_res_band.Draw("2 same")

    for y, style in [(0.0, 1), (-1.0, 2), (1.0, 2)]:
        line = ROOT.TLine(obs_binning.lowBound(), y, obs_binning.highBound(), y)
        persistent_objects.append(line)
        line.SetLineColor(ROOT.kGray + 2)
        line.SetLineStyle(style)
        line.Draw()
    h_res.Draw("PE same")


ROOT.gStyle.SetOptStat(0)
c_pre_post = ROOT.TCanvas("c_pre_post", "Pre-fit and post-fit model comparison", int(0.6 * 1500), int(0.6 * 720))
persistent_objects.append(c_pre_post)

pad_geometry = [
    ("p1t", 0.0, 0.32, 0.5, 1.0),
    ("p1b", 0.0, 0.0, 0.5, 0.32),
    ("p2t", 0.5, 0.32, 1.0, 1.0),
    ("p2b", 0.5, 0.0, 1.0, 0.32),
]
pads = {}
for name, x0, y0, x1, y1 in pad_geometry:
    pad = ROOT.TPad(name, "", x0, y0, x1, y1)
    persistent_objects.append(pad)
    pad.SetLeftMargin(0.20 if "1" in name else 0.06)
    pads[name] = pad
    if "t" in name:  # top pads: no x labels, little bottom margin
        pad.SetBottomMargin(0.025)
        pad.SetTopMargin(0.08)
    else:  # bottom pads: room for the x axis
        pad.SetTopMargin(0.05)
        pad.SetBottomMargin(0.42)
    pad.Draw()

mu_hat, mu_err = poi_var.getVal(), poi_var.getError()
draw_fit_panel(pads["p1t"], pads["p1b"], "Pre-fit", prefit_total, prefit_samples,
               prefit_sigma, "#mu = 1.0 (nominal)")
draw_fit_panel(pads["p2t"], pads["p2b"], "Post-fit", postfit_total, postfit_samples,
               postfit_sigma, f"#mu = {mu_hat:.2f} #pm {mu_err:.2f}",
               residual_title="")

c_pre_post.Draw()


CMS Combine-style pulls (unformtunately broken with jsroot):

In [ ]:
%jsroot off

# more plotting code for the pulls...

# -------------------------------------------------
# Nuisance-parameter pulls, combine style
#
# For every constrained nuisance parameter theta, the plot shows
#     pull = (theta_hat - theta_0) / sigma_0
# with the post-fit error in units of the pre-fit constraint width
# sigma_0: well-behaved parameters sit at 0 +/- 1, i.e. the central
# value inside the dark grey band and the error bar shorter than the
# light grey band means the parameter was constrained by the data.
# The POI on top has no pre-fit constraint, so it is shown as its
# compatibility with the nominal value 1 in units of its own error.
# -------------------------------------------------

def pretty_param_label(name):
    """Combine-style axis label for a workspace parameter name."""
    if name == poi_var.GetName():
        return "#mu (signal strength)"
    if name.startswith("alpha_"):
        return "#alpha_{" + name[len("alpha_"):] + "}"
    if name.startswith("gamma_stat_") and name.endswith(tuple(str(i) for i in range(10))):
        bin_index = name.rsplit("_", 1)[1]
        return f"#gamma_{{stat}}^{{bin {int(bin_index) + 1}}}"
    return name


def pull_rank(name):
    """Sort order for the pulls plot: POI first, then systematics, then
    MC-stat. parameters."""
    if name == poi_var.GetName():
        return 0
    if name.startswith("alpha_"):
        return 1
    return 2


# (theta - theta_0) / sigma_0 with the post-fit error in units of sigma_0.
# The POI has no pre-fit constraint: quote its compatibility with the
# nominal value 1 in units of its own post-fit error.
pull_points = []
final_pars = {p.GetName(): (p.getVal(), p.getError()) for p in fit_result_hf.floatParsFinal()}
for name, (fit_val, fit_err) in final_pars.items():
    if name == poi_var.GetName():
        pull_points.append((name, (fit_val - 1.0) / fit_err, 1.0))
    else:
        nominal, sigma0 = prefit_pars[name]
        pull_points.append((name, (fit_val - nominal) / sigma0, fit_err / sigma0))
pull_points.sort(key=lambda row: (pull_rank(row[0]), row[0]))

n_pulls = len(pull_points)
x_max = max(3.2, max(abs(pull) + err + 0.6 for _n, pull, err in pull_points))

c_pulls = ROOT.TCanvas("c_pulls", "Nuisance-parameter pulls", 720, 140 + 60 * n_pulls)
persistent_objects.append(c_pulls)
c_pulls.SetLeftMargin(0.24)
c_pulls.SetRightMargin(0.03)

pull_axes = ROOT.TH2F("pull_axes", "", 10, -x_max, x_max, n_pulls, 0.5, n_pulls + 0.5)
persistent_objects.append(pull_axes)
pull_axes.SetStats(0)
pull_axes.GetXaxis().SetTitle("(#theta - #theta_{0}) / #sigma_{0}")
pull_axes.GetXaxis().SetTitleSize(0.05)
pull_axes.GetXaxis().SetTitleOffset(1.0)
for i, (name, _pull, _err) in enumerate(pull_points):  # top row = first entry
    pull_axes.GetYaxis().SetBinLabel(n_pulls - i, pretty_param_label(name))
pull_axes.GetYaxis().SetLabelSize(0.06)
pull_axes.Draw()

# grey reference boxes for |pull| < 2 and |pull| < 1
for half_width, color in [(2.0, ROOT.kGray), (1.0, ROOT.kGray + 1)]:
    box = ROOT.TBox(-half_width, 0.5, half_width, n_pulls + 0.5)
    persistent_objects.append(box)
    box.SetFillColor(color)
    box.Draw()

zero_line = ROOT.TLine(0.0, 0.5, 0.0, n_pulls + 0.5)
persistent_objects.append(zero_line)
zero_line.SetLineColor(ROOT.kGray + 3)
zero_line.SetLineStyle(ROOT.kDashed)
zero_line.Draw()

pull_graph = ROOT.TGraphErrors(n_pulls)
persistent_objects.append(pull_graph)
for i, (_name, pull, err) in enumerate(pull_points):
    pull_graph.SetPoint(i, pull, n_pulls - i)
    pull_graph.SetPointError(i, err, 0.0)
pull_graph.SetMarkerStyle(21)
pull_graph.SetMarkerSize(1.1)
pull_graph.SetLineWidth(2)
pull_graph.Draw("PE")

c_pulls.Draw()

print("pulls (postfit - prefit) / sigma_prefit:")
for name, pull, err in pull_points:
    print(f"   {name:32s} {pull:+.3f} +/- {err:.3f}")

**Exercise question:** why is the signal strength pulled the most?

### Congratulations

Congrats 🎉! You made it to the end of this very long notebook.
    
Have some cake: 🎂


### Higher-level frameworks in the wild

The takeaway: binned likelihood fits are typically built from Monte Carlo templates that are interpolated according to nuisance parameters, and such models are complicated enough that nobody writes them out by hand. Higher-level frameworks generate them from a declarative specification.

We have seen a simple example with HistFactory, but the LHC experiments often use their own high-level frameworks for this:

- [CMS Combine](https://cms-analysis.github.io/HiggsAnalysis-CombinedLimit/): the CMS statistics framework, built on top of RooFit and RooStats
- [TRExFitter](https://trexfitter-docs.web.cern.ch/trexfitter/): widely used in ATLAS, generates HistFactory workspaces from configuration files

They differ in configuration language and features, but underneath they all describe the same kind of binned likelihood model that you now know how to build, inspect, and fit yourself.